# NetPyNE: Cortical Column Simulation

Build a multi-layer cortical microcircuit with morphologically detailed neurons using NetPyNE + NEURON.

## Prerequisites
- Understanding of cortical microcircuit architecture (excitatory/inhibitory populations)
- Familiarity with compartmental neuron models and Hodgkin-Huxley dynamics
- Knowledge of synaptic models (excitatory and inhibitory conductances)
- Basic Python skills

## Setup
Run the cell below to install dependencies and download data.

In [ ]:
!pip install netpyne neuron matplotlib

In [ ]:
from netpyne import specs, sim
import matplotlib.pyplot as plt

# Network parameters
netParams = specs.NetParams()
simConfig = specs.SimConfig()

# --- Cell types ---
netParams.cellParams['PYR'] = {
    'secs': {'soma': {'geom': {'diam': 18.8, 'L': 18.8},
                      'mechs': {'hh': {'gnabar': 0.12, 'gkbar': 0.036, 'gl': 0.003, 'el': -70}}}}
}
netParams.cellParams['INT'] = {
    'secs': {'soma': {'geom': {'diam': 10, 'L': 10},
                      'mechs': {'hh': {'gnabar': 0.12, 'gkbar': 0.036, 'gl': 0.003, 'el': -70}}}}
}

# --- Populations ---
netParams.popParams['E'] = {'cellType': 'PYR', 'numCells': 80}
netParams.popParams['I'] = {'cellType': 'INT', 'numCells': 20}

# --- Synaptic model ---
netParams.synMechParams['exc'] = {'mod': 'Exp2Syn', 'tau1': 0.1, 'tau2': 5.0, 'e': 0}
netParams.synMechParams['inh'] = {'mod': 'Exp2Syn', 'tau1': 0.1, 'tau2': 10.0, 'e': -70}

# --- Connectivity ---
netParams.connParams['E->E'] = {'preConds': {'pop': 'E'}, 'postConds': {'pop': 'E'},
    'weight': 0.005, 'delay': 2, 'synMech': 'exc', 'probability': 0.1}
netParams.connParams['E->I'] = {'preConds': {'pop': 'E'}, 'postConds': {'pop': 'I'},
    'weight': 0.01, 'delay': 2, 'synMech': 'exc', 'probability': 0.2}
netParams.connParams['I->E'] = {'preConds': {'pop': 'I'}, 'postConds': {'pop': 'E'},
    'weight': 0.02, 'delay': 2, 'synMech': 'inh', 'probability': 0.3}

# --- Background input ---
netParams.stimSourceParams['bkg'] = {'type': 'NetStim', 'rate': 50, 'noise': 0.5}
netParams.stimTargetParams['bkg->E'] = {'source': 'bkg', 'conds': {'pop': 'E'},
    'weight': 0.01, 'delay': 1, 'synMech': 'exc'}

# --- Simulation config ---
simConfig.duration = 500
simConfig.dt = 0.025
simConfig.recordTraces = {'V_soma': {'sec': 'soma', 'loc': 0.5, 'var': 'v'}}
simConfig.analysis['plotRaster'] = {'saveFig': False}
simConfig.analysis['plotTraces'] = {'include': [0, 1, 2], 'saveFig': False}

# --- Run ---
sim.createSimulateAnalyze(netParams=netParams, simConfig=simConfig)

## References

- Dura-Bernal, S., Suter, B. A., Gleeson, P., Cantarelli, M., Quintana, A., Rodriguez, F., ... & Bhatt, D. K. (2019). NetPyNE, a tool for data-driven multiscale modeling of brain circuits. *eLife*, 8, e44494. https://doi.org/10.7554/eLife.44494
- Hines, M. L., & Carnevale, N. T. (1997). The NEURON simulation environment. *Neural Computation*, 9(6), 1179\u20131209.
- Markram, H., Muller, E., Ramaswamy, S., Reimann, M. W., Abdellah, M., Sanchez, C. A., ... & Schuermann, F. (2015). Reconstruction and simulation of neocortical microcircuitry. *Cell*, 163(2), 456\u2013492.
- Potjans, T. C., & Diesmann, M. (2014). The cell-type specific cortical microcircuit: relating structure and activity in a full-scale spiking network model. *Cerebral Cortex*, 24(3), 785\u2013806.